In [1]:
import json
from tqdm import tqdm

In [2]:
with open('korean-audio-text-develop.json') as fopen:
    rows = json.load(fopen)
len(rows)

5139

In [3]:
mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 5139/5139 [00:00<00:00, 3279252.74it/s]


5139

In [5]:
import faiss
import os
import numpy as np

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'korean-audio-text-develop/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 5139/5139 [00:00<00:00, 5781.92it/s]


In [6]:
len(data)

5139

In [7]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [9]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[:10]

{'audio_filename': ['korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_0.mp3',
  'korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_1.mp3',
  'korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_2.mp3',
  'korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_3.mp3',
  'korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_4.mp3',
  'korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_5.mp3',
  'korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_6.mp3',
  'korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_7.mp3',
  'korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_8.mp3',
  'korean-audio-text-develop_audio/korean-audio-text-develop-data-train-00003-of-00005_9.mp3'],
 'text': ['거의 40만 이상 되죠',
  '이때

In [10]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'korean-audio-text-develop')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 412.54ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  234kB /  234kB,  586kB/s  
New Data Upload: 100%|██████████|  234kB /  234kB,  586kB/s  
Processing Files (1 / 1): 100%|██████████|  234kB /  234kB,  390kB/s  
New Data Upload: 100%|██████████|  234kB /  234kB,  390kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.11s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/96831507aacf8c3d69c8c58918df13e41d8cf31e', commit_message='Upload dataset', commit_description='', oid='96831507aacf8c3d69c8c58918df13e41d8cf31e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)